# 03 · Deep ensemble under corruption (upgrade, Colab T4)

Three ResNet-50 members on the same leak-free split, differing only in training seed:
**42 is the notebook 01 baseline, reused**; 43 and 44 are trained here (about 26 min each).
The ensemble predicts the mean softmax of the members. It goes through the same sweep as the
single model (4 corruptions × severities 0–5; Wilson and bootstrap intervals; coverage and
AURC for max softmax, entropy, margin, and the members' mutual information), and one figure
compares it with the single model at every severity.

**Needs:** notebook 01 finished (its seed 42 run in `OUT`). Notebook 02 is not needed; its
sweep is reused if it is there. **After a disconnect:** run all again. A member or sweep that
is already in Drive is reused, so only unfinished work is repeated.

In [ ]:
# ---- settings: the only cell you normally edit ---------------------------------
REPO_URL = "https://github.com/brycedaniliuc/COMP2035-grape-calibration.git"
REPO_REF = "main"                       # a branch, tag or commit hash
ZIP_PATH = "/content/drive/MyDrive/grape_dataset_for_colab.zip"   # must match Drive
DATA_ROOT = "/content/grape_dataset"    # the zip extracts to this folder
OUT = "/content/drive/MyDrive/COMP2035_runs"   # every output is written here, as it goes
MEMBER_SEEDS = (42, 43, 44)   # 42 = notebook 01's baseline, reused

In [ ]:
import os, subprocess, sys, time
from pathlib import Path
T0 = time.time()

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")

REPO = Path("/content/COMP2035-grape-calibration")
if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet", "--tags"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_REF], check=True)
commit = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], check=True,
                        capture_output=True, text=True).stdout.strip()
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("code at", REPO_REF, commit)

# Colab already has torch, torchvision, numpy, scikit-learn, scikit-image and matplotlib.
# Do NOT pip install requirements.txt here: it pins the CPU build of torch.
import importlib.util
missing = [m for m in ("skimage", "sklearn", "nbformat") if importlib.util.find_spec(m) is None]
if missing:
    pkg = {"skimage": "scikit-image", "sklearn": "scikit-learn"}
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    *[pkg.get(m, m) for m in missing]], check=True)

from src.train import environment, select_device
import json
env = environment(select_device())
print(json.dumps(env, indent=1))
if env["device"] != "cuda":
    print("!! NO GPU: Runtime > Change runtime type > T4 GPU, then run all again")
os.makedirs(OUT, exist_ok=True)
print("outputs ->", OUT)

In [ ]:
# Fail early: unzip and check the exact per-folder counts BEFORE any training.
import shutil, zipfile
if not os.path.isdir(DATA_ROOT):
    assert os.path.exists(ZIP_PATH), (
        f"{ZIP_PATH} not found. Put grape_dataset_for_colab.zip at the top level of My Drive. "
        f"Files there: {sorted(os.listdir('/content/drive/MyDrive'))[:40]}")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall("/content/")
from src.data import check_supplied_dataset
check_supplied_dataset(DATA_ROOT)       # raises unless 7,222 train + 1,805 test, per class
print(f"elapsed {(time.time() - T0) / 60:.1f} min")

## The seed 42 baseline from notebook 01

In [ ]:
from src.ensemble import check_cached, ensemble_sweep, ensure_member, ensure_sweep
from src.ensemble import find_run
BASE = find_run(OUT, seed=42)
assert BASE, f"no finished resnet50 source run in {OUT}: run notebook 01 first"
print("baseline:", BASE.name)
# before any training: a cached member sweep from before D20 is brought up to date from its
# saved logits (CPU only), or this stops with the reason
check_cached(BASE, MEMBER_SEEDS)

## Members (about 26 min for each one trained)

A member already in Drive is reused. Each checkpoint is saved to Drive as it improves.

In [ ]:
MEMBERS = []
for seed in MEMBER_SEEDS:
    MEMBERS.append(ensure_member(BASE, seed, data_root=DATA_ROOT))
    print(f"seed {seed}: {MEMBERS[-1].name} | elapsed {(time.time() - T0) / 60:.1f} min")

## Each member through the degradation sweep (about 6 min each, estimated)

In [ ]:
for m in MEMBERS:
    ensure_sweep(m, data_root=DATA_ROOT)
print(f"elapsed {(time.time() - T0) / 60:.1f} min")

## The ensemble: mean softmax across members, in every condition

In [ ]:
from src.evaluate import print_threshold
ENS = ensemble_sweep(MEMBERS, data_root=DATA_ROOT)
ENS_PATH = f"{OUT}/ensemble_{ENS['ens_id']}.json"
print_threshold(ENS["threshold"])

## Single model vs ensemble at every severity (paper PDF + slide PNG), to Drive

In [ ]:
from src.figures import render_ensemble
paths = render_ensemble(ENS_PATH, f"{OUT}/figures_{ENS['ens_id']}", tag=ENS["ens_id"])
print(len(paths), "files ->", f"{OUT}/figures_{ENS['ens_id']}")

## Verify (Tier 1 must pass): every member, then the ensemble

In [ ]:
for m in MEMBERS:
    r = subprocess.run([sys.executable, "-m", "src.verify", "--run", str(m), "--require-sweep"],
                       capture_output=True, text=True)
    print(m.name, "|", r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-500:])
    assert r.returncode == 0, r.stdout[-6000:]
r = subprocess.run([sys.executable, "-m", "src.verify", "--ensemble", ENS_PATH],
                   capture_output=True, text=True)
print(r.stdout[-6000:], r.stderr[-3000:])
assert r.returncode == 0, "a Tier 1 check failed: stop and investigate"
print(f"total elapsed {(time.time() - T0) / 60:.1f} min")

## Everything this notebook relies on or wrote, in Drive

In [ ]:
ids = [ENS["ens_id"]] + [json.loads(m.read_text())["run_id"] for m in MEMBERS]
for p in sorted(Path(OUT).rglob("*")):
    if p.is_file() and any(i in p.name for i in ids):
        print(f"{p.stat().st_size / 1e6:9.2f} MB  {p.relative_to(OUT)}")